1. Basic Tasks
1. Connect Power BI Desktop to a Databricks SQL warehouse via Partner Connect (or a manual
connection) and build one report against a gold table.
2. Create a Unity Catalog connection to an external PostgreSQL (or MySQL) database and a foreign
catalog exposing one of its tables.
3. Read about Lakebase and write a short summary of when you'd reach for it instead of a Delta table.

#BASIC

#1
![](/Workspace/Users/sakshijain7945@gmail.com/DE_Assignment/day-11/b1.jpeg)

In [0]:

-- 2
-- We can do it through Databricks Lakehouse Federation.

CREATE CONNECTION neon_db_connection
TYPE postgresql
OPTIONS (
  host 'ep-floral-haze-at0pddgx-pooler.c-9.us-east-1.aws.neon.tech',
  port '5432',
  user '<username>',
  password 'password'
);



In [0]:

CREATE FOREIGN CATALOG neon_foreign_catalog
USING CONNECTION neon_db_connection
OPTIONS (database 'neondb');



In [0]:
select * from neon_foreign_catalog.public.patients

#3 

Databricks Lakebase is a fully managed, serverless PostgreSQL-compatible database designed specifically for operational (OLTP) workloads. 
- we prefer Lakebase over Delta tables : 
- when we require single-digit millisecond latency 
- Your application serves thousands of simultaneous users, web applications, or active AI agents writing states concurrently.


#INTERMEDIATE

#4 
Publish your Power BI report to the Power BI service and document the two ways data could go stale for this connection type (scheduled refresh vs. live query).

Sheduled refresh - 
In an Import connection model that relies on a scheduled refresh, data goes stale because it is not updated in real-time.
Information stays outdated until the next scheduled time arrives.

In a Live Query - 
the report queries the data source in real-time. However, data can still appear stale due to caching behaviors.
Data can still look old because of caching behaviors built into the system or browser.

In [0]:
-- 5
SELECT 
    n.customer_id,
    n.name,
    n.email,
    f.blood_type,
    f.gender
FROM 
    cyntexa_dev.sales.customer AS n
JOIN 
    neon_foreign_catalog.public.patients AS f
    ON n.customer_id = f.patient_id;


In [0]:
EXPLAIN EXTENDED
SELECT 
    n.customer_id,
    n.name,
    n.email,
    f.blood_type,
    f.gender
FROM 
    cyntexa_dev.sales.customer AS n
JOIN 
    neon_foreign_catalog.public.patients AS f
    ON n.customer_id = f.patient_id;


#6

create share gold_total_revenue_share
comment 'Sharing gold sales total revenue table with partner';

alter share gold_total_revenue_share
add table cyntexa_dev.sales.gold_total_revenue;

- Step 1 — create the recipient
create recipient partner_recipient
using id '<recipient-sharing-identifier>';

- Step 2 — grant access
grant select on share gold_total_revenue_share to recipient partner_recipient;

- Step 3 — verify the share was created correctly
show shares;
show grants on share gold_total_revenue_share;

#7
![](/Workspace/Users/sakshijain7945@gmail.com/DE_Assignment/day-11/7.jpeg)

#ADVANCED

#8

The data-sharing decision matrix for Cyntexa based on the Databricks Unity Catalog Delta Sharing capabilities.In Databricks Delta Sharing, there are two primary sharing flows:
- **Databricks-to-Databricks sharing**: A seamless, native sharing flow where the recipient accesses the shared data directly through their own Unity Catalog-enabled Databricks workspace. It supports advanced assets like volumes, views, and AI models.

- **Databricks-to-Open sharing**: A secure protocol where the recipient uses an open-source Delta Sharing client (like Pandas, Apache Spark, Power BI, etc.) via a secure credential file. It primarily supports tabular data and standard views.

#9

Query federation stops making sense when data-freshness requirements are loose (allowing stale data) or when query volume and complexity become heavy.

Data-Freshness Conditions : 
- Nightly updates are acceptable: If the business users or analytics dashboards only require data that is updated once a day, querying the live operational Postgres database in real time offers no added value.
- Historical reporting focus: When analytical tasks focus on day-over-day or month-over-month trends rather than minute-by-minute updates, a nightly batch pipeline provides a stable snapshot without risking live database performance.

Query-Volume Conditions : 
- High concurrent query volume: As the number of analytical queries increases, query federation repeatedly hits the operational Postgres instance. This can exhaust database connections and severely degrade application performance.
- Large data scan volume: Analytical queries often scan millions of rows or perform heavy aggregations. Running these directly or federated against an operational transactional (OLTP) database can spike CPU/memory usage and cause outages.

#10
Propose an LTAP architecture for a new Cyntexa feature (e.g., a real-time inventory-check app) that
needs both OLTP writes (Lakebase) and OLAP analytics (Lakehouse) on the same data, specifying
what syncs where and who owns each side operationally.

OLTP side (Lakebase):
App writes here directly — every inventory update, stock check, order placed Fast inserts/updates/deletes, low latency, handles concurrent app traffic Owned operationally by the application/engineering team — they manage schema, app logic, and uptime for live writes

OLAP side (Lakehouse / Delta tables):
Used for reporting, dashboards, trend analysis (e.g. "which products are running low across all stores") Not written to directly by the app — receives data through sync Owned operationally by the data engineering/analytics team — they manage the gold tables, dashboards, and downstream pipelines
Lakebase automatically syncs operational changes into managed Delta tables (via its built-in change data feed) No manual ETL job needed just to move data — sync happens continuously/near real-time Delta tables then feed into silver/gold tables, dashboards, and Genie for business-facing analytics
